Data Preperation


In [9]:
# Install missing packages into the active notebook kernel (safe to re-run)
import importlib
import subprocess
import sys

REQUIRED = ["seaborn"]
missing = [pkg for pkg in REQUIRED if importlib.util.find_spec(pkg) is None]

if missing:
    print(f"Installing missing packages: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required plotting packages are already installed.")

All required plotting packages are already installed.


In [10]:
# Standard library
import glob
import warnings
warnings.filterwarnings('ignore')

# Numerics / data
import numpy as np
import pandas as pd

# ── Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Sklearn utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ── Global settings
SEED      = 42
np.random.seed(SEED)

plt.rcParams.update({
    'figure.dpi'     : 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size'      : 11,
})

print('All imports successful.')
print(f'pandas  {pd.__version__}  |  numpy  {np.__version__}')

All imports successful.
pandas  2.2.2  |  numpy  2.0.2


In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Loading datasets


In [12]:
!gdown --output DemoStats_2025_CSV_fixed.zip https://drive.google.com/uc?id=1NoxxXG9bgK7Ky8lbHrj2Ik6aPuECUcq4
!gdown --output HouseholdSpend_2025_CSV_fixed.zip https://drive.google.com/uc?id=1NWRKLuetADT_LfJR7m6KLPbNuiD0j4mp

Downloading...
From (original): https://drive.google.com/uc?id=1NoxxXG9bgK7Ky8lbHrj2Ik6aPuECUcq4
From (redirected): https://drive.google.com/uc?id=1NoxxXG9bgK7Ky8lbHrj2Ik6aPuECUcq4&confirm=t&uuid=5b6d948f-2a17-459e-a5b2-495650a5e09d
To: /content/DemoStats_2025_CSV_fixed.zip
100% 504M/504M [00:07<00:00, 70.6MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1NWRKLuetADT_LfJR7m6KLPbNuiD0j4mp
From (redirected): https://drive.google.com/uc?id=1NWRKLuetADT_LfJR7m6KLPbNuiD0j4mp&confirm=t&uuid=9676e2cd-9859-4d42-80b3-1cbbf683bc63
To: /content/HouseholdSpend_2025_CSV_fixed.zip
100% 2.07G/2.07G [00:48<00:00, 42.4MB/s]


In [ ]:
import io
import os
import zipfile
from pathlib import Path
import pandas as pd

# ── FILE PATHS (already good) ────────────────────────────────────────────────
DEMO_ZIP = "/content/DemoStats_2025_CSV_fixed.zip"
SPEND_ZIP = "/content/HouseholdSpend_2025_CSV_fixed.zip"

DOC_KEYWORDS = ("metadata", "variables list", "variables change list", "release notes")

# ── HELPERS ──────────────────────────────────────────────────────────────────
def should_skip_csv(path_in_zip: str):
    lower = path_in_zip.lower()
    base = os.path.basename(lower)

    if "documentation/" in lower:
        return True, "inside Documentation/"
    if not lower.endswith(".csv"):
        return True, "not a CSV"
    if any(k in base for k in DOC_KEYWORDS):
        return True, "documentation/metadata CSV"
    return False, ""


def is_data_csv(path_in_zip: str):
    lower = path_in_zip.lower()
    base = os.path.basename(lower)
    return base.endswith("_geo.csv") or "_geo_p" in base


def read_csv_with_geo_filter(file_obj, encoding="latin-1", geo_value="PRCDDA", max_chunks=10):
    collected = []
    chunk_count = 0

    for chunk in pd.read_csv(
        file_obj,
        encoding=encoding,
        low_memory=False,
        chunksize=20000  # r
    ):
        if "GEO" in chunk.columns:
            chunk = chunk[chunk["GEO"] == geo_value]

        if not chunk.empty:
            collected.append(chunk)

        chunk_count += 1
        if chunk_count >= max_chunks:
            break

    if not collected:
        return pd.DataFrame()

    return pd.concat(collected, ignore_index=True)


def load_dataset(zip_path: str, label: str):
    print(f"\nOpening: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")

    combined = None

    with zipfile.ZipFile(zip_path, "r") as zf:
        for entry in zf.infolist():
            name = entry.filename

            skip, reason = should_skip_csv(name)
            if skip or not is_data_csv(name):
                continue

            try:
                with zf.open(entry) as f:
                    df = read_csv_with_geo_filter(f)

                print(f"  Loaded: {os.path.basename(name)} {df.shape}")

                if combined is None:
                    combined = df
                else:
                    #
                    combined = pd.concat([combined, df], ignore_index=True)

            except Exception as e:
                print(f"  Failed: {name} → {e}")

    if combined is None:
        raise ValueError(f"No usable data found for {label}")

    print(f"→ Final shape for {label}: {combined.shape}")
    return combined


# ── RUN ──────────────────────────────────────────────────────────────────────
print("=" * 60)
print("Loading DemoStats …")
print("=" * 60)
demo_raw = load_dataset(DEMO_ZIP, "DemoStats")

print("\n" + "=" * 60)
print("Loading HouseholdSpend …")
print("=" * 60)
spend_raw = load_dataset(SPEND_ZIP, "HouseholdSpend")

Loading DemoStats …

Opening: /content/DemoStats_2025_CSV_fixed.zip (504.5 MB)
  Loaded: DemoStats_(DST)_2020_2025_GEO.csv (57936, 468)
  Loaded: DemoStats_(DST)_2025_2025_GEO.csv (0, 0)
  Loaded: DemoStats_(DST)_2028_2025_GEO.csv (57936, 468)


In [ ]:
#inspect data structure
print('=== DemoStats (first 5 rows) ===')
display(demo_raw.head())
print('\nColumn names (first 20):', demo_raw.columns.tolist()[:20])

print('\n=== HouseholdSpend (first 5 rows) ===')
display(spend_raw.head())
print('\nColumn names (first 20):', spend_raw.columns.tolist()[:20])

In [ ]:
# Expected column names for geographic ID and level
GEO_ID_COL  = 'CODE'   # numeric PRCDDA code
GEO_LVL_COL = 'GEO'   # text level label, e.g. 'PRCDDA'

# Verify they exist in both datasets
for name, df in [('DemoStats', demo_raw), ('HouseholdSpend', spend_raw)]:
    print(f'{name}: {GEO_ID_COL} in columns → {GEO_ID_COL in df.columns}')
    print(f'{name}: {GEO_LVL_COL} in columns → {GEO_LVL_COL in df.columns}')
    if GEO_LVL_COL in df.columns:
        print(f'  Unique GEO levels: {df[GEO_LVL_COL].unique()}')
    print()

FILTER TO PRCDDA LEVEL

In [ ]:
TARGET_GEO = 'PRCDDA'

demo_da  = demo_raw[demo_raw[GEO_LVL_COL]  == TARGET_GEO].copy()
spend_da = spend_raw[spend_raw[GEO_LVL_COL] == TARGET_GEO].copy()

print(f'DemoStats   – PRCDDA rows : {len(demo_da):,}  |  columns: {demo_da.shape[1]}')
print(f'HouseholdSpend – PRCDDA rows: {len(spend_da):,}  |  columns: {spend_da.shape[1]}')

# Sanity check: each CODE should appear once per dataset after filtering
demo_dups  = demo_da[GEO_ID_COL].duplicated().sum()
spend_dups = spend_da[GEO_ID_COL].duplicated().sum()
print(f'\nDuplicate CODE entries → DemoStats: {demo_dups}  |  HouseholdSpend: {spend_dups}')

CONSTRUCTING TARGET VARIABLE (HBR)

In [ ]:
# ── Verify the required columns exist in the spending dataset ─────────────────
required = ['HSSH001S', 'HSAGDISPIN']
missing  = [c for c in required if c not in spend_da.columns]
if missing:
    raise KeyError(f'Missing columns in HouseholdSpend PRCDDA data: {missing}')
print('Required columns confirmed present.')

# ── Compute HBR ───────────────────────────────────────────────────────────────
spend_da['y'] = spend_da['HSSH001S'] / spend_da['HSAGDISPIN']

# Rows where HSAGDISPIN == 0 produce inf; rows with NaN in either produce NaN
n_inf = np.isinf(spend_da['y']).sum()
n_nan = spend_da['y'].isna().sum()
print(f'Computed y:  {len(spend_da):,} rows | inf: {n_inf} | NaN: {n_nan}')

# Drop rows where y is undefined (zero income or missing values)
spend_da = spend_da[np.isfinite(spend_da['y'])].copy()
print(f'After removing inf/NaN: {len(spend_da):,} rows remain.')

REMOVE LEAKAGE VARIABLE

In [ ]:
# ── Define leakage variable sets ──────────────────────────────────────────────
LEAKAGE_EXACT = {
    'HSAGDISPIN', 'HSAGDISCIN', 'HSHNIAGG',
    'HSTT001', 'HSTE001', 'HSTX001', 'HSTC001',
    'HSSH001S',   # numerator of y – must also be removed
}

# All columns starting with HSSH (shelter sub-items)
hssh_cols = [c for c in spend_da.columns if c.startswith('HSSH')]
print(f'HSSH columns found: {len(hssh_cols)}')

all_leakage = LEAKAGE_EXACT | set(hssh_cols)
print(f'Total leakage columns to drop: {len(all_leakage)}')

# ── Drop from spending dataframe (before merge) ───────────────────────────────
cols_to_drop_spend = [c for c in all_leakage if c in spend_da.columns]
spend_clean = spend_da.drop(columns=cols_to_drop_spend)

print(f'\nspend_da columns before drop : {spend_da.shape[1]}')
print(f'spend_clean columns after drop: {spend_clean.shape[1]}')
print(f'Columns removed: {len(cols_to_drop_spend)}')

MODELLING TABLEAU

In [ ]:
# ── Inner join on PRCDDA code ─────────────────────────────────────────────────
tableau = pd.merge(
    spend_clean,
    demo_da,
    on   = GEO_ID_COL,
    how  = 'inner',
    suffixes = ('_spend', '_demo')
)

print(f'Tableau dimensions (rows × cols): {tableau.shape}')
print(f'Number of unique PRCDDA codes   : {tableau[GEO_ID_COL].nunique():,}')

In [ ]:
#inspecting the tableau
geo_duplicates = [c for c in tableau.columns if c.endswith('_spend') or c.endswith('_demo')
                  if c.startswith('GEO')]
if geo_duplicates:
    tableau = tableau.drop(columns=geo_duplicates[1:])  # keep first copy
    tableau = tableau.rename(columns={geo_duplicates[0]: GEO_LVL_COL})

print('Tableau column dtypes summary:')
display(tableau.dtypes.value_counts())

print('\nMissing values per column (top 20 by missingness):')
miss = tableau.isnull().mean().sort_values(ascending=False)
display(miss[miss > 0].head(20))

# cleaning the tableau

ID_COLS    = [GEO_ID_COL, GEO_LVL_COL]   # non-predictive metadata
TARGET_COL = 'y'

# All numeric columns (excluding IDs and target)
feature_cols = [
    c for c in tableau.select_dtypes(include='number').columns
    if c not in ID_COLS and c != TARGET_COL
]

print(f'Feature columns available for modelling: {len(feature_cols)}')
print(f'Target column                           : {TARGET_COL}')
print(f'Identifier columns (excluded)           : {ID_COLS}')

In [ ]:
#HANDLING MISSING VALUES
HIGH_MISS_THRESHOLD = 0.50

# Compute missingness on feature columns
miss_rate  = tableau[feature_cols].isnull().mean()
drop_cols  = miss_rate[miss_rate > HIGH_MISS_THRESHOLD].index.tolist()
keep_cols  = miss_rate[miss_rate <= HIGH_MISS_THRESHOLD].index.tolist()

print(f'Columns dropped (>{HIGH_MISS_THRESHOLD*100:.0f}% missing): {len(drop_cols)}')
print(f'Columns retained                                   : {len(keep_cols)}')

feature_cols = keep_cols   # update list
tableau      = tableau[ID_COLS + [TARGET_COL] + feature_cols].copy()

In [ ]:
# REMOVE 0 VARIANVE COLUMN
std_vals   = tableau[feature_cols].std()
zero_var   = std_vals[std_vals == 0].index.tolist()
feature_cols = [c for c in feature_cols if c not in zero_var]
tableau      = tableau[ID_COLS + [TARGET_COL] + feature_cols].copy()

print(f'Zero-variance columns removed : {len(zero_var)}')
print(f'Remaining feature columns     : {len(feature_cols)}')
print(f'Final tableau shape           : {tableau.shape}')

train test split

In [ ]:
# Bin y into deciles for stratification (ensures representative splits)
y_bins = pd.qcut(tableau[TARGET_COL], q=10, labels=False, duplicates='drop')

X = tableau[feature_cols]
y = tableau[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size   = 0.20,
    random_state= SEED,
    stratify    = y_bins
)

# Also preserve the full tableau indices aligned with splits
train_idx = X_train.index
test_idx  = X_test.index

print(f'Training set : {X_train.shape[0]:,} rows  ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Test set     : {X_test.shape[0]:,}  rows  ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'Features     : {X_train.shape[1]}')

In [ ]:
#median imputation
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns = feature_cols,
    index   = X_train.index
)
X_test_imp  = pd.DataFrame(
    imputer.transform(X_test),
    columns = feature_cols,
    index   = X_test.index
)

remaining_nans = X_train_imp.isnull().sum().sum() + X_test_imp.isnull().sum().sum()
print(f'Remaining NaNs after imputation: {remaining_nans}')

In [ ]:
#standardise features
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_imp),
    columns = feature_cols,
    index   = X_train_imp.index
)
X_test_scaled  = pd.DataFrame(
    scaler.transform(X_test_imp),
    columns = feature_cols,
    index   = X_test_imp.index
)

print(f'X_train_scaled : mean ≈ {X_train_scaled.mean().mean():.4f}  |  std ≈ {X_train_scaled.std().mean():.4f}')
print(f'X_test_scaled  : mean ≈ {X_test_scaled.mean().mean():.4f}   |  std ≈ {X_test_scaled.std().mean():.4f}')